# 🔬 Notebook 3 — Ablation Study: What Actually Works?

**Goal**: Isolate the contribution of each component in the training recipe.  
**Model**: ResNet-18 (lightweight, ~11M params vs 23M for ResNet-50)  
**Resolution**: 224×224 (fast on CPU)  
**Epochs**: 3 per experiment  

### Experiment Design (Additive)

Each experiment inherits everything from the previous one and adds **one new component**:

| # | Experiment | Optimizer | Loss | Augmentation | Preprocess |
|---|-----------|-----------|------|-------------|------------|
| 1 | Vanilla | SGD (lr=0.01) | CrossEntropy | ✗ | Simple resize |
| 2 | + AdamW | **AdamW** (lr=1e-4) | CrossEntropy | ✗ | Simple resize |
| 3 | + Focal Loss | AdamW | **Focal Loss** | ✗ | Simple resize |
| 4 | + Augmentation | AdamW | Focal Loss | **✓** | Simple resize |
| 5 | + Ben Graham | AdamW | Focal Loss | ✓ | **Ben Graham** |

---

## 1. Setup

In [1]:
import sys, os, json, time, copy
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet18_Weights
from tqdm.notebook import tqdm
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, roc_auc_score,
    classification_report
)
from sklearn.model_selection import StratifiedKFold

import albumentations as A
from albumentations.pytorch import ToTensorV2

from config import (
    APTOS_TRAIN_CSV, APTOS_TRAIN_IMAGES,
    IMAGENET_MEAN, IMAGENET_STD, RANDOM_SEED, NUM_CLASSES,
    DR_GRADES, DR_COLORS, LOG_DIR, FIGURES_DIR,
    seed_everything, setup_directories,
)
from loss import FocalLoss, compute_class_weights
from preprocessing import ben_graham_preprocess

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ── Ablation-specific config ──
IMG_SIZE = 224         # small for speed
BATCH_SIZE = 8         # fits 16 GB CPU RAM with 224×224
EPOCHS = 3             # quick ablation
NUM_WORKERS = 0

seed_everything(RANDOM_SEED)
setup_directories()

DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}  |  Image size: {IMG_SIZE}  |  Epochs: {EPOCHS}')

Device: cpu  |  Image size: 224  |  Epochs: 3


## 2. Data Utilities

In [2]:
# ── Stratified train/val split ──
def get_train_val_split(df, val_fold=0, n_folds=5, seed=RANDOM_SEED):
    """Stratified K-fold split, returns (train_df, val_df)."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df, df['diagnosis'])):
        if fold_idx == val_fold:
            return df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)
    return None, None


# ── Lightweight Dataset (supports toggling Ben Graham on/off) ──
class AblationDataset(Dataset):
    """Configurable dataset for ablation experiments.

    Args:
        df: DataFrame with 'id_code' and 'diagnosis'.
        image_dir: Path to APTOS images.
        transform: Albumentations Compose pipeline.
        use_ben_graham: If True, apply Ben Graham preprocessing.
        img_size: Target resolution.
    """

    def __init__(self, df, image_dir, transform, use_ben_graham=False, img_size=224):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.use_ben_graham = use_ben_graham
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_dir / f"{row['id_code']}.png"
        image = cv2.imread(str(img_path))

        if self.use_ben_graham:
            image = ben_graham_preprocess(image, self.img_size)
        else:
            # Simple resize only
            image = cv2.resize(image, (self.img_size, self.img_size))

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        augmented = self.transform(image=image)
        image = augmented['image']

        return image, int(row['diagnosis'])


# ── Transform builders ──
def get_simple_transform(img_size):
    """Val/test transform: just normalize."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_augmented_transform(img_size):
    """Training transform: geometric + color augmentation."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=360, p=0.7),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print('Data utilities ready.')

Data utilities ready.


## 3. Model Builder

In [3]:
def build_resnet18(num_classes=NUM_CLASSES, pretrained=True):
    """Build a plain ResNet-18 with a new head."""
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    in_features = model.fc.in_features  # 512 for ResNet-18
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, num_classes),
    )
    return model


# Quick check
test_model = build_resnet18(pretrained=False)
total_params = sum(p.numel() for p in test_model.parameters())
print(f'ResNet-18 params: {total_params:,}')
del test_model

ResNet-18 params: 11,179,077


## 4. Train & Validate Functions

In [4]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch_num, total_epochs):
    """Train for one epoch. Returns (loss, accuracy)."""
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f'Epoch {epoch_num+1}/{total_epochs} [Train]', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        all_preds.extend(logits.detach().argmax(1).numpy())
        all_labels.extend(labels.numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return running_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate. Returns (loss, acc, kappa, auc, preds, labels, probs)."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        probs = F.softmax(logits, dim=1).numpy()
        all_probs.append(probs)
        all_preds.extend(logits.argmax(1).numpy())
        all_labels.extend(labels.numpy())

    all_probs = np.vstack(all_probs)
    all_labels_arr = np.array(all_labels)
    all_preds_arr = np.array(all_preds)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    # Binary referable DR AUC
    binary_labels = (all_labels_arr >= 2).astype(int)
    binary_probs = all_probs[:, 2:].sum(axis=1)
    try:
        epoch_auc = roc_auc_score(binary_labels, binary_probs)
    except ValueError:
        epoch_auc = 0.0

    return epoch_loss, epoch_acc, epoch_kappa, epoch_auc


print('Train/val functions ready.')

Train/val functions ready.


## 5. Experiment Runner

In [5]:
def run_experiment(name, train_loader, val_loader, criterion, optimizer_fn, epochs=EPOCHS):
    """Run a single ablation experiment.

    Args:
        name: Experiment label.
        train_loader: Training DataLoader.
        val_loader: Validation DataLoader.
        criterion: Loss function.
        optimizer_fn: Callable(model.parameters()) -> optimizer.
        epochs: Number of training epochs.

    Returns:
        dict with best metrics and training history.
    """
    seed_everything(RANDOM_SEED)
    model = build_resnet18(pretrained=True).to(DEVICE)
    optimizer = optimizer_fn(model.parameters())
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Use CE on CPU for val regardless (simpler comparison)
    val_criterion = nn.CrossEntropyLoss()

    best_kappa = -1.0
    best_metrics = {}
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_kappa': [], 'val_auc': []}

    print(f'\n{"="*55}')
    print(f'  Experiment: {name}')
    print(f'{"="*55}')

    start_time = time.time()
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, epoch, epochs)
        v_loss, v_acc, v_kappa, v_auc = validate(model, val_loader, val_criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        history['val_kappa'].append(v_kappa)
        history['val_auc'].append(v_auc)

        print(f'  Epoch {epoch+1}/{epochs} — '
              f'Loss: {t_loss:.4f}/{v_loss:.4f}  '
              f'Acc: {t_acc:.3f}/{v_acc:.3f}  '
              f'κ: {v_kappa:.4f}  AUC: {v_auc:.4f}')

        if v_kappa > best_kappa:
            best_kappa = v_kappa
            best_metrics = {
                'name': name,
                'best_epoch': epoch + 1,
                'val_acc': v_acc,
                'val_kappa': v_kappa,
                'val_auc': v_auc,
                'val_loss': v_loss,
            }

    elapsed = time.time() - start_time
    best_metrics['time_seconds'] = round(elapsed, 1)
    best_metrics['history'] = history
    print(f'  Done in {elapsed:.0f}s — Best κ: {best_kappa:.4f}')
    return best_metrics


print('Experiment runner ready.')

Experiment runner ready.


## 6. Prepare Data

In [6]:
df = pd.read_csv(APTOS_TRAIN_CSV)
train_df, val_df = get_train_val_split(df, val_fold=0)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}')

# Class weights for Focal Loss (computed once)
train_labels = torch.tensor(train_df['diagnosis'].values)
alpha_weights = compute_class_weights(train_labels, NUM_CLASSES)
print(f'Alpha weights: {alpha_weights.numpy().round(3)}')

Train: 2929  |  Val: 733
Alpha weights: [0.406 1.979 0.733 3.804 2.482]


## 7. Run All Experiments

In [7]:
results = []

# ── Shared transforms ──
simple_tf = get_simple_transform(IMG_SIZE)
aug_tf = get_augmented_transform(IMG_SIZE)

# ── Shared val loader (simple resize, no augmentation, no Ben Graham) ──
val_ds_simple = AblationDataset(val_df, str(APTOS_TRAIN_IMAGES), simple_tf,
                                use_ben_graham=False, img_size=IMG_SIZE)
val_loader_simple = DataLoader(val_ds_simple, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=NUM_WORKERS)

# ── Val loader with Ben Graham ──
val_ds_graham = AblationDataset(val_df, str(APTOS_TRAIN_IMAGES), simple_tf,
                                use_ben_graham=True, img_size=IMG_SIZE)
val_loader_graham = DataLoader(val_ds_graham, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=NUM_WORKERS)

### Exp 1 — Vanilla: SGD + CrossEntropy + No Augmentation + Simple Resize

In [8]:
train_ds = AblationDataset(train_df, str(APTOS_TRAIN_IMAGES), simple_tf,
                           use_ben_graham=False, img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS)

results.append(run_experiment(
    name='1. Vanilla (SGD + CE)',
    train_loader=train_loader,
    val_loader=val_loader_simple,
    criterion=nn.CrossEntropyLoss(),
    optimizer_fn=lambda p: optim.SGD(p, lr=0.01, momentum=0.9),
))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ADMIN/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:04<00:00, 11.5MB/s]


  Experiment: 1. Vanilla (SGD + CE)


Epoch 1/3 [Train]:   0%|          | 0/367 [00:00<?, ?it/s]

  Epoch 1/3 — Loss: 1.4432/1.0015  Acc: 0.594/0.569  κ: 0.3606  AUC: 0.8713


Epoch 2/3 [Train]:   0%|          | 0/367 [00:00<?, ?it/s]

  Epoch 2/3 — Loss: 1.0183/0.8369  Acc: 0.649/0.690  κ: 0.6067  AUC: 0.9040


Epoch 3/3 [Train]:   0%|          | 0/367 [00:00<?, ?it/s]

  Epoch 3/3 — Loss: 0.8818/0.7887  Acc: 0.697/0.714  κ: 0.6845  AUC: 0.9110
  Done in 724s — Best κ: 0.6845


### Exp 2 — + AdamW

In [9]:
train_ds = AblationDataset(train_df, str(APTOS_TRAIN_IMAGES), simple_tf,
                           use_ben_graham=False, img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS)

results.append(run_experiment(
    name='2. + AdamW',
    train_loader=train_loader,
    val_loader=val_loader_simple,
    criterion=nn.CrossEntropyLoss(),
    optimizer_fn=lambda p: optim.AdamW(p, lr=1e-4, weight_decay=1e-4),
))


  Experiment: 2. + AdamW


Epoch 1/3 [Train]:   0%|          | 0/367 [00:00<?, ?it/s]

  Epoch 1/3 — Loss: 0.7884/0.5493  Acc: 0.721/0.787  κ: 0.8520  AUC: 0.9815


Epoch 2/3 [Train]:   0%|          | 0/367 [00:00<?, ?it/s]

  Epoch 2/3 — Loss: 0.5267/0.4894  Acc: 0.814/0.828  κ: 0.8811  AUC: 0.9804


Epoch 3/3 [Train]:   0%|          | 0/367 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Exp 3 — + Focal Loss

In [ ]:
train_ds = AblationDataset(train_df, str(APTOS_TRAIN_IMAGES), simple_tf,
                           use_ben_graham=False, img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS)

results.append(run_experiment(
    name='3. + Focal Loss',
    train_loader=train_loader,
    val_loader=val_loader_simple,
    criterion=FocalLoss(gamma=2.0, alpha=alpha_weights),
    optimizer_fn=lambda p: optim.AdamW(p, lr=1e-4, weight_decay=1e-4),
))

### Exp 4 — + Augmentation

In [ ]:
train_ds = AblationDataset(train_df, str(APTOS_TRAIN_IMAGES), aug_tf,
                           use_ben_graham=False, img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS)

results.append(run_experiment(
    name='4. + Augmentation',
    train_loader=train_loader,
    val_loader=val_loader_simple,
    criterion=FocalLoss(gamma=2.0, alpha=alpha_weights),
    optimizer_fn=lambda p: optim.AdamW(p, lr=1e-4, weight_decay=1e-4),
))

### Exp 5 — + Ben Graham Preprocessing

In [ ]:
# Training with Ben Graham + augmentation
train_ds = AblationDataset(train_df, str(APTOS_TRAIN_IMAGES), aug_tf,
                           use_ben_graham=True, img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS)

results.append(run_experiment(
    name='5. + Ben Graham',
    train_loader=train_loader,
    val_loader=val_loader_graham,
    criterion=FocalLoss(gamma=2.0, alpha=alpha_weights),
    optimizer_fn=lambda p: optim.AdamW(p, lr=1e-4, weight_decay=1e-4),
))

## 8. Results Comparison

In [ ]:
# ── Summary table ──
summary_rows = []
for r in results:
    summary_rows.append({
        'Experiment': r['name'],
        'Best Epoch': r['best_epoch'],
        'Val Acc': f"{r['val_acc']:.4f}",
        'Val QWK': f"{r['val_kappa']:.4f}",
        'Val AUC': f"{r['val_auc']:.4f}",
        'Time (s)': r['time_seconds'],
    })

summary_df = pd.DataFrame(summary_rows)
print('\n📊 Ablation Study Results')
print('=' * 85)
print(summary_df.to_string(index=False))
print('=' * 85)

In [ ]:
# ── Bar chart comparison ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
exp_names = [r['name'] for r in results]
short_names = [n.split('. ')[1] for n in exp_names]

# QWK
kappas = [r['val_kappa'] for r in results]
bars = axes[0].bar(range(len(results)), kappas, color='mediumpurple', edgecolor='white')
for i, (b, k) in enumerate(zip(bars, kappas)):
    axes[0].text(i, k + 0.005, f'{k:.3f}', ha='center', fontweight='bold', fontsize=9)
axes[0].set_xticks(range(len(results)))
axes[0].set_xticklabels(short_names, rotation=30, ha='right', fontsize=9)
axes[0].set_title('Quadratic Weighted Kappa (↑)', fontweight='bold')
axes[0].set_ylim(0, max(kappas) * 1.15)
axes[0].grid(axis='y', alpha=0.3)

# AUC
aucs = [r['val_auc'] for r in results]
bars = axes[1].bar(range(len(results)), aucs, color='teal', edgecolor='white')
for i, (b, a) in enumerate(zip(bars, aucs)):
    axes[1].text(i, a + 0.003, f'{a:.3f}', ha='center', fontweight='bold', fontsize=9)
axes[1].set_xticks(range(len(results)))
axes[1].set_xticklabels(short_names, rotation=30, ha='right', fontsize=9)
axes[1].set_title('Referable DR AUC (↑)', fontweight='bold')
axes[1].set_ylim(0, max(aucs) * 1.08)
axes[1].grid(axis='y', alpha=0.3)

# Accuracy
accs = [r['val_acc'] for r in results]
bars = axes[2].bar(range(len(results)), accs, color='steelblue', edgecolor='white')
for i, (b, a) in enumerate(zip(bars, accs)):
    axes[2].text(i, a + 0.005, f'{a:.3f}', ha='center', fontweight='bold', fontsize=9)
axes[2].set_xticks(range(len(results)))
axes[2].set_xticklabels(short_names, rotation=30, ha='right', fontsize=9)
axes[2].set_title('Accuracy (↑)', fontweight='bold')
axes[2].set_ylim(0, max(accs) * 1.15)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Ablation Study — Component Contribution (ResNet-18, 224×224, 3 epochs)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_study_bars.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Training curves overlay (val kappa per epoch) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_list = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']

for i, r in enumerate(results):
    label = r['name'].split('. ')[1]
    ep_range = range(1, len(r['history']['val_kappa']) + 1)
    axes[0].plot(ep_range, r['history']['val_kappa'], 'o-',
                 color=colors_list[i], label=label, linewidth=2, markersize=6)
    axes[1].plot(ep_range, r['history']['val_loss'], 's-',
                 color=colors_list[i], label=label, linewidth=2, markersize=6)

axes[0].set_title('Val QWK per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Quadratic κ')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_title('Val Loss per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (CE)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle('Ablation Study — Training Curves', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_study_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Save Results

In [ ]:
# Save ablation results (without heavy history for clean JSON)
save_results = []
for r in results:
    entry = {k: v for k, v in r.items() if k != 'history'}
    save_results.append(entry)

results_path = LOG_DIR / 'ablation_study_results.json'
with open(results_path, 'w') as f:
    json.dump(save_results, f, indent=2)
print(f'Results saved: {results_path}')

## 10. Summary

This ablation isolates the contribution of each training recipe component:

| Component | What it Does |
|-----------|-------------|
| **AdamW** | Adaptive LR per parameter + decoupled weight decay → faster, more stable convergence vs SGD |
| **Focal Loss** | Down-weights easy (majority class) samples → forces model to learn minority classes |
| **Augmentation** | Geometric + color transforms → regularizes, prevents overfitting on small dataset |
| **Ben Graham** | Removes illumination variation, enhances lesion contrast → cleaner signal for the model |

**Interpretation tips:**
- If QWK jumps significantly at a step → that component matters the most.
- If augmentation *hurts* in 3 epochs → it's because augmented images are harder; with more epochs it would help.
- If Ben Graham helps → the preprocessing pipeline is pulling its weight.

**Next step:** Use the full recipe with CBAM-ResNet50 at 512×512 on GPU for the real training.